In [1]:
import os
import pandas as pd

print('Libraries loaded.')
pd.set_option('display.expand_frame_repr', False)

Libraries loaded.


In [2]:
RUN_LABEL   = 'global'
RUN_VERSION = 'v6'
RUN_NAME    = f'{RUN_LABEL}_{RUN_VERSION}'

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'

fire_metrics_path = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'fire_metrics', f'master_{RUN_NAME}.csv')
fire_metrics_clean_path = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'inputs', 'fire_metrics_clean.csv')
anomaly_path      = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'inputs', 'era5_anomalies_by_lag.csv')
save_dir          = os.path.join(BASE_OUT_DIR, 'runs', RUN_NAME, 'inputs')

print(f'Fire metrics       : {fire_metrics_path}')
print(f'Fire metrics clean : {fire_metrics_clean_path}')
print(f'Anomaly file       : {anomaly_path}')
print(f'Save dir           : {save_dir}')

Fire metrics       : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\fire_metrics\master_global_v6.csv
Fire metrics clean : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\inputs\fire_metrics_clean.csv
Anomaly file       : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\inputs\era5_anomalies_by_lag.csv
Save dir           : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\inputs


In [3]:
fire_metrics       = pd.read_csv(fire_metrics_path)
fire_metrics_clean = pd.read_csv(fire_metrics_clean_path)
anomaly_df         = pd.read_csv(anomaly_path)

print(f'fire_metrics       : {fire_metrics.shape}')
print(f'fire_metrics_clean : {fire_metrics_clean.shape}')
print(f'anomaly_df         : {anomaly_df.shape}')

fire_metrics       : (14885, 44)
fire_metrics_clean : (14878, 44)
anomaly_df         : (14878, 14)


In [4]:
# Merge
fire_metrics_final = fire_metrics_clean.merge(anomaly_df, on=['eco_id', 'year'], how='left')

# Assert no rows lost or duplicated
assert len(fire_metrics_final) == len(fire_metrics_clean), \
    f'Row count mismatch: {len(fire_metrics_clean)} → {len(fire_metrics_final)}'

# Assert no unexpected NaN in climate columns (every fire row should have climate data)
climate_cols = [c for c in anomaly_df.columns if 'anomaly' in c]
n_missing = fire_metrics_final[climate_cols].isna().any(axis=1).sum()
assert n_missing == 0, f'{n_missing} rows have missing climate data after merge'

print(f'Rows before : {len(fire_metrics_clean):,}')
print(f'Rows after  : {len(fire_metrics_final):,}')
print(fire_metrics_final.head())

Rows before : 14,878
Rows after  : 14,878
   eco_id                        eco_name  biome_num                                      biome_name  year  onset_doy  peak_doy  end_doy  season_length  n_detections  ...  window_days_30d  temp_anomaly_45d  precip_anomaly_45d  window_days_45d  temp_anomaly_60d  precip_anomaly_60d  window_days_60d  temp_anomaly_90d  precip_anomaly_90d  window_days_90d
0       1  Albertine Rift montane forests          1  Tropical & Subtropical Moist Broadleaf Forests  2003         35       193      278            244         26772  ...               30         -0.148986            0.258227               45         -0.137756            0.789682               60         -0.175300            0.788038               90
1       1  Albertine Rift montane forests          1  Tropical & Subtropical Moist Broadleaf Forests  2004         43       186      274            232         36852  ...               30          0.118395           -0.329260               45          

In [5]:
# Select and validate final columns for analysis dataset
# window_days_* columns excluded here but available in era5_anomalies_by_lag.csv for QC if needed

keep_cols = [
    # identifiers
    'eco_id', 'eco_name', 'biome_num', 'biome_name', 'year',
    # primary fire timing metrics
    'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    # detection volume
    'n_detections',
    # quality flags
    'pct_years_valid', 'bimodal_flag_eco', 'frac_flagged',
    'mean_profile_corr', 'cv_peak_doy',
    # climate predictors (pre-onset lagged anomalies)
    'temp_anomaly_30d', 'precip_anomaly_30d',
    'temp_anomaly_45d', 'precip_anomaly_45d',
    'temp_anomaly_60d', 'precip_anomaly_60d',
    'temp_anomaly_90d', 'precip_anomaly_90d',
]

# Verify all expected columns exist before selecting
missing_cols = [c for c in keep_cols if c not in fire_metrics_final.columns]
if missing_cols:
    print(f'WARNING: {len(missing_cols)} expected columns not found: {missing_cols}')
else:
    print('All expected columns present.')

fire_metrics_final = fire_metrics_final[keep_cols]

print(f'\nFinal shape : {fire_metrics_final.shape[0]:,} rows × {fire_metrics_final.shape[1]} columns')
print(f'\nMissing values:')
print(fire_metrics_final.isnull().sum())

All expected columns present.

Final shape : 14,878 rows × 23 columns

Missing values:
eco_id                 0
eco_name               0
biome_num              0
biome_name             0
year                   0
onset_doy              0
peak_doy               0
end_doy                0
season_length          0
n_detections           0
pct_years_valid        0
bimodal_flag_eco       0
frac_flagged           0
mean_profile_corr      0
cv_peak_doy           10
temp_anomaly_30d       0
precip_anomaly_30d     0
temp_anomaly_45d       0
precip_anomaly_45d     0
temp_anomaly_60d       0
precip_anomaly_60d     0
temp_anomaly_90d       0
precip_anomaly_90d     0
dtype: int64


In [6]:
# Save
out_path = os.path.join(save_dir, 'analysis_dataset.csv')
fire_metrics_final.to_csv(out_path, index=False)
print(f'Saved {fire_metrics_final.shape[0]:,} rows × {fire_metrics_final.shape[1]} columns to:')
print(out_path)

Saved 14,878 rows × 23 columns to:
C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v6\inputs\analysis_dataset.csv
